# 第 3 堂　相似，不等於能回答

[![在 Colab 開啟](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rthtsai/junior-lab/blob/main/w03.ipynb)

**怎麼用**：由上往下，一格一格按左邊的 ▶ 執行。

- 看到 **✏️** 的格子，要自己把 `____` 換成答案。沒填就按，會出現紅色錯誤，這是正常的。
- 卡住的順序：先看格子裡的提示 → 問同組 → 再舉手。
- 上週程式學會排序。今天讓程式多做一件事：判斷這一段**能不能回答**題目，並指出**是哪一句**支持它。

## 先做這一格（一開始就按）

下載語意模型要 1 到 3 分鐘，先讓它在背景跑。
如果跳出「要重新啟動工作階段」，按「重新啟動」再繼續，不用重按這格。

In [ ]:
!pip install -q sentence-transformers
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("BAAI/bge-m3")
print("語意模型準備好了")

---
## 第 1 節　資料與分句

題目一個，五段短文。每一段裡有一句或幾句，用「。」分開。
下面已經寫好：把一段文字拆成句子。

In [ ]:
question = "為什麼冰塊會浮在水上？"
passages = {
    "A": "水結成冰時，分子會排成比較鬆散的六角形結構，體積變大約百分之九。所以同樣重量的冰比水輕，密度比較小，才會浮在水面上。",
    "B": "把同樣大小的一杯水和一杯冰分別秤重，冰會比較輕。原因是水結冰時體積膨脹，裡面多出了許多空隙。",
    "C": "飲料店的杯子裡總是放滿冰塊，冰塊浮在水面上，一搖杯子就會碰撞出清脆的聲音。",
    "D": "冰塊放久了會融化，因為室溫比較高，熱量傳進冰塊，所以杯子外面會結出水珠。",
    "E": "北極熊的毛看起來是白色的，其實每一根毛是透明中空的，因為這種構造，牠在雪地中不容易被發現。"
}

def split_sentences(text):
    return [s for s in text.split("。") if s]

for name, text in passages.items():
    print(name, "共", len(split_sentences(text)), "句")

---
## 第 2 節　規則關：有沒有「原因線索詞」

題目問「為什麼」，能回答的句子通常會說出原因，像「因為」「所以」。
先寫一個規則：句子裡有沒有線索詞。

### ✏️ 1　有沒有線索詞

`for c in CUES` 會把線索詞一個一個拿出來。請填空，讓它檢查「這個線索詞有沒有出現在句子裡」。

提示：迴圈變數是哪一個？

In [ ]:
CUES = ["因為", "所以", "由於", "因此", "原因", "導致", "造成", "使得"]

def has_cue(sentence):
    return any(____ in sentence for c in CUES)

for name, text in passages.items():
    cue = any(has_cue(s) for s in split_sentences(text))
    print(name, "有線索詞" if cue else "沒有線索詞")

### 想一想（回表單）

只用規則的話，哪一段**有線索詞，卻其實答不到題目**？

---
## 第 3 節　語意關：這一句和題目像不像

規則只看有沒有「因為」，不知道句子在講什麼。再加一關：用上週的方法，算每一句和題目有多像。
下面這個函式已經寫好，你只要會用它。

In [ ]:
def sentence_sims(question, sentences):
    q = model.encode([question], normalize_embeddings=True)
    s = model.encode(sentences, normalize_embeddings=True)
    return [float(x) for x in (s @ q.T).ravel()]

for name, text in passages.items():
    sents = split_sentences(text)
    sims = sentence_sims(question, sents)
    best = max(range(len(sents)), key=lambda i: sims[i])
    print(name, "最像的一句：", round(sims[best], 3), sents[best][:18], "…")

### ✏️ 2　兩關都要過

`judge` 會把一段拆成句子，只留下有線索詞的句子，再看其中**和題目最像的那句**。
規則：有線索詞，**而且**分數達到門檻 `TH`，才算「能回答」。

提示：兩個條件都要成立，中間要放哪一個運算詞？（`and`、`or`、`not` 三選一）

In [ ]:
TH = 0.66   # 門檻：分數要到這個數字才算夠像

def judge(question, text):
    sentences = split_sentences(text)
    sims = sentence_sims(question, sentences)
    cands = [(sim, s) for sim, s in zip(sims, sentences) if has_cue(s)]   # 只留有線索詞的句子
    best_sim, best_s = max(cands) if cands else (0.0, None)
    can = (len(cands) > 0) ____ (best_sim >= TH)
    return {"can_answer": can,
            "evidence": best_s if can else None,
            "sim": round(best_sim, 3) if cands else None}

def show(name, r):
    if r["can_answer"]:
        print(name, "能回答", r["sim"], "依據：", r["evidence"])
    else:
        print(name, "不能回答", r["sim"])

for name, text in passages.items():
    show(name, judge(question, text))

### 想一想（回表單）

1. 有哪一段和題目**很像**，卻被判成不能回答？是被哪一關擋下來的？
2. 如果**只看語意分數、不要規則關**，會多放進哪一段？為什麼它會騙到？

---
## 應用任務　三個題組，算兩個數字

每個題組有一個題目、五到六段短文。`ans` 是能回答的段落，`ev` 是支持它的那一句裡的關鍵片段。
下面兩格已經寫好，**只要按執行**，看結果。

In [ ]:
SETS = {
    "題組 1　冰塊": {
        "q": "為什麼冰塊會浮在水上？",
        "ans": [
            "A",
            "B"
        ],
        "ev": {
            "A": "所以同樣重量的冰比水輕",
            "B": "原因是水結冰時體積膨脹"
        },
        "p": {
            "A": "水結成冰時，分子會排成比較鬆散的六角形結構，體積變大約百分之九。所以同樣重量的冰比水輕，密度比較小，才會浮在水面上。",
            "B": "把同樣大小的一杯水和一杯冰分別秤重，冰會比較輕。原因是水結冰時體積膨脹，裡面多出了許多空隙。",
            "C": "飲料店的杯子裡總是放滿冰塊，冰塊浮在水面上，一搖杯子就會碰撞出清脆的聲音。",
            "D": "冰塊放久了會融化，因為室溫比較高，熱量傳進冰塊，所以杯子外面會結出水珠。",
            "E": "北極熊的毛看起來是白色的，其實每一根毛是透明中空的，因為這種構造，牠在雪地中不容易被發現。"
        }
    },
    "題組 2　手機發燙": {
        "q": "為什麼手機用久了會變燙？",
        "ans": [
            "A",
            "B"
        ],
        "ev": {
            "A": "電能有一部分會變成熱",
            "B": "熱量不容易散出去"
        },
        "p": {
            "A": "手機的處理器運算時，需要電流通過電路。電能有一部分會變成熱，所以玩遊戲或錄影時，手機會明顯發燙。",
            "B": "充電時電池內部的化學反應也會產生熱。加上手機機身又薄又密閉，熱量不容易散出去，因此越用越燙。",
            "C": "手機很燙的時候，有些人會把它放進冰箱降溫，但這樣做可能讓內部結露而損壞。",
            "D": "手機電池會越用越不耐用，原因是每次充放電，電池裡的材料都會有一點損耗。",
            "E": "夏天戶外溫度高，所以冰淇淋融化得特別快，吃的時候要快一點。"
        }
    },
    "題組 3　飛機窗戶": {
        "q": "為什麼飛機的窗戶是圓角的？",
        "ans": [
            "A",
            "B"
        ],
        "ev": {
            "A": "方角處會集中應力",
            "B": "圓角能把壓力平均分散到整個窗框"
        },
        "p": {
            "A": "早期的飛機窗戶是方形的。高空中機艙內外壓力不同，方角處會集中應力，造成金屬疲勞和裂縫，所以後來改成圓角。",
            "B": "圓角能把壓力平均分散到整個窗框，因此不容易裂開。這個設計來自早期飛行事故的調查。",
            "C": "搭飛機時，很多人喜歡選靠窗的座位，可以看到雲海、日出和城市夜景，窗戶的形狀也各有不同。",
            "D": "飛機的窗戶其實有三層，最外層承受壓力，中間層有一個小孔平衡氣壓，內層則是保護板。",
            "E": "飛機窗戶的小孔能讓水氣散出去，所以窗戶不容易起霧。",
            "F": "颱風來襲時，航空公司常因為強風而取消班機，旅客可以申請改票或退票。"
        }
    }
}

In [ ]:
def run_set(name, S, verbose=True):
    right = 0
    hit = 0
    can_total = 0
    for n, text in S["p"].items():
        r = judge(S["q"], text)
        gold = n in S["ans"]
        ok = (r["can_answer"] == gold)
        right += ok
        if gold:
            can_total += 1
            if r["can_answer"] and S["ev"][n] in r["evidence"]:
                hit += 1
        if verbose:
            mark = "✔" if ok else "✘"
            ans = "能" if r["can_answer"] else "不能"
            print(" ", mark, n, ans, r["sim"], ("依據：" + r["evidence"]) if r["evidence"] else "")
    print(name, "判斷正確率", right, "/", len(S["p"]), "　依據命中率", hit, "/", can_total)
    return right, len(S["p"]), hit, can_total

total_right = total_n = 0
for name, S in SETS.items():
    r, n, h, t = run_set(name, S)
    total_right += r; total_n += n
print("三題組合計", total_right, "/", total_n)

### 要交的（填在表單）

1. 三個題組的**判斷正確率**各是多少？
2. 哪一段「有線索詞、同主題，但答的是別的問題」？程式有被它騙到嗎？
3. 用一句話說：為什麼「相似」不等於「能回答」？

---
## 下週要交（回家做）

### 一　調門檻

把 `TH` 改成 0.35、0.55、0.70，各跑一次，記下三題組合計的正確率。
想想：門檻太低、太高，各會錯在哪裡？

In [ ]:
for t in [0.35, 0.55, 0.70]:
    TH = t
    right = n_all = 0
    for name, S in SETS.items():
        r, n, h, tt = run_set(name, S, verbose=False)
        right += r; n_all += n
    print("TH =", t, "→ 合計", right, "/", n_all)
TH = 0.66   # 記得改回來

### 二　自己出一題

照格式寫一個「為什麼」型題目、六段短文，指定兩段能回答，並寫下支持句的關鍵片段。
**至少放兩種干擾段**：①同主題，但沒說原因（沒有「因為」）；②有「因為／所以」，但答的是別的問題。
寫完執行，記下每段結果與判斷正確率。

In [ ]:
my_set = {
    "q": "在這裡寫你的題目",
    "ans": ["A", "B"],
    "ev": {"A": "A 段支持句的關鍵片段", "B": "B 段支持句的關鍵片段"},
    "p": {
        "A": "能回答 1。",
        "B": "能回答 2。",
        "C": "干擾段：同主題但沒說原因。",
        "D": "干擾段：有因為／所以，但答別的問題。",
        "E": "干擾段 3。",
        "F": "干擾段 4。",
    },
}
run_set("我的題組", my_set)

### 三　改一次

改一次 `CUES` 或 `TH`，記下前後正確率各是多少，有沒有變好。

最後：**File（檔案）→ 在雲端硬碟中儲存副本**，存到自己的雲端硬碟。